In [ ]:
import os, gc, time, json, warnings
import numpy as np
import pandas as pd
from itertools import combinations, product
warnings.filterwarnings("ignore")

from scipy.optimize import minimize
from scipy.special  import softmax

from sklearn.model_selection    import StratifiedKFold
from sklearn.preprocessing      import StandardScaler, label_binarize, normalize
from sklearn.decomposition      import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection  import RFE
from sklearn.calibration        import CalibratedClassifierCV
from sklearn.metrics            import (accuracy_score, balanced_accuracy_score,
                                        f1_score, matthews_corrcoef,
                                        precision_score, recall_score,
                                        confusion_matrix, roc_auc_score,
                                        average_precision_score)
from sklearn.linear_model       import LogisticRegression
from sklearn.svm                import SVC, LinearSVC
from sklearn.neighbors          import KNeighborsClassifier
from sklearn.neural_network     import MLPClassifier
from sklearn.naive_bayes        import GaussianNB
from sklearn.ensemble           import (RandomForestClassifier, ExtraTreesClassifier,
                                        AdaBoostClassifier,
                                        HistGradientBoostingClassifier)
from sklearn.tree               import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from lightgbm                   import LGBMClassifier
from xgboost                    import XGBClassifier
from catboost                   import CatBoostClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone, is_classifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import log_loss

In [ ]:

FEATURES_PATH = "/kaggle/input/datasets/hsharmaa/foodev-v3-10299/features_prott5_10639.csv"  # TRAIN, full 1024-d
OUTPUT_DIR    = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ID_COL      = "seq_id"
TARGET_COL  = "primary_label"
STRAT_COL   = "secondary_label"
META_COLS   = [ID_COL, "primary_label", "secondary_label"]
CLASS_NAMES = {0: "Non_EV", 1: "Milk_EV", 2: "Plant_EV"}


SELECTION_METHOD = "LGBM_gain"
SELECT_K         = 384        
BASE_POOL        = ["LightGBM", "SVM_RBF", "XGBoost", "MLP", "KNN_cosine"]
COMBO_SIZES      = (3, 4, 5)

SEED     = 42
N_OUTER  = 5
N_INNER  = 3
INNER_VAL_FRAC = 0.20

CALIBRATE          = True     
ENRICH_META        = True     
DECISION_OBJECTIVE = "acc"    

OUTPUT_FILE = os.path.join(OUTPUT_DIR, "stack_base_combinations_strategyA.csv")

In [ ]:


def select_lgbm_gain(Xtr, Xte, ytr, k=SELECT_K, seed=SEED):
    k = min(k, Xtr.shape[1])
    if k >= Xtr.shape[1]:
        return Xtr, Xte
    est = LGBMClassifier(n_estimators=300, max_depth=6, importance_type="gain",
                         class_weight="balanced", verbosity=-1,
                         random_state=seed, n_jobs=-1)
    s = SelectFromModel(est, max_features=k, threshold=-np.inf).fit(Xtr, ytr)
    return s.transform(Xtr), s.transform(Xte)

In [ ]:


class BalancedWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator=None):
        self.estimator = estimator
    def fit(self, X, y, sample_weight=None):
        w = compute_sample_weight("balanced", y) if sample_weight is None else sample_weight
        self.estimator_ = clone(self.estimator).fit(X, y, sample_weight=w)
        self.classes_ = self.estimator_.classes_
        return self
    def predict_proba(self, X):
        return self.estimator_.predict_proba(X)
    def predict(self, X):
        return self.estimator_.predict(X)



def make_model(name):
    if name == "LightGBM":
        return LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=6,
                              num_leaves=15, min_child_samples=40,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                              reg_lambda=5.0, class_weight="balanced",
                              verbosity=-1, random_state=SEED, n_jobs=-1)
    if name == "XGBoost":
        return BalancedWrapper(XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
            colsample_bytree=0.5, reg_lambda=5.0, min_child_weight=5,
            eval_metric="mlogloss", tree_method="hist",
            random_state=SEED, n_jobs=-1))
    if name == "SVM_RBF":

        return SVC(C=5.0, kernel="rbf", gamma="scale",
                   probability=not CALIBRATE, class_weight="balanced",
                   random_state=SEED)
    if name == "MLP":
        return MLPClassifier(hidden_layer_sizes=(512, 256, 128), alpha=1e-3,
                             max_iter=500, early_stopping=True,
                             n_iter_no_change=15,
                             validation_fraction=INNER_VAL_FRAC,
                             random_state=SEED)
    if name == "KNN_cosine":

        return KNeighborsClassifier(n_neighbors=25, weights="distance",
                                    metric="cosine", n_jobs=-1)
    raise ValueError(f"No factory entry for model '{name}'")


def fit_predict_base(name, A_tr, y_tr, B_list):
    est = make_model(name)
    if CALIBRATE:
        est = CalibratedClassifierCV(est, method="isotonic", cv=3)
    est.fit(A_tr, y_tr)
    out = []
    for B in B_list:
        P = est.predict_proba(B)
        full = np.zeros((len(B), 3))          
        for j, c in enumerate(est.classes_):
            full[:, int(c)] = P[:, j]
        out.append(full)
    return out

In [ ]:


def get_metrics(y_true, y_pred, y_score=None, n_classes=3):
    m = {
        "acc" : accuracy_score(y_true, y_pred),
        "bacc": balanced_accuracy_score(y_true, y_pred),
        "f1"  : f1_score(y_true, y_pred, average="macro"),
        "pre" : precision_score(y_true, y_pred, average="macro", zero_division=0),
        "sens": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc" : matthews_corrcoef(y_true, y_pred),
    }
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    specs, npvs = [], []
    for i in range(n_classes):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)
        specs.append(tn / (tn + fp) if (tn + fp) else 0.0)
        npvs.append(tn / (tn + fn) if (tn + fn) else 0.0)
        m[f"sens_c{i}"] = tp / (tp + fn) if (tp + fn) else 0.0
        m[f"spec_c{i}"] = specs[-1]
        m[f"pre_c{i}"]  = tp / (tp + fp) if (tp + fp) else 0.0
        m[f"npv_c{i}"]  = npvs[-1]
    m["spec"] = float(np.mean(specs))
    m["npv"]  = float(np.mean(npvs))
    if y_score is not None:
        yb = label_binarize(y_true, classes=list(range(n_classes)))
        try:
            m["auc"] = roc_auc_score(y_true, y_score, multi_class="ovr",
                                     average="macro", labels=list(range(n_classes)))
        except Exception:
            m["auc"] = np.nan
        try:
            m["ap"] = average_precision_score(yb, y_score, average="macro")
        except Exception:
            m["ap"] = np.nan
    else:
        m["auc"] = m["ap"] = np.nan
    return m


def fmt(a, raw=False):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    if a.size == 0:
        return "n/a"
    s = 1 if raw else 100
    return f"{a.mean()*s:.2f} \u00b1 {a.std()*s:.2f}"


def fmt_ci(a, raw=False):
    """Mean with a 95% CI across folds (normal approximation on the fold SE)."""
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    if a.size == 0:
        return "n/a"
    s  = 1 if raw else 100
    mu = a.mean() * s
    se = a.std(ddof=1) / np.sqrt(a.size) * s if a.size > 1 else 0.0
    return f"{mu:.2f} [{mu-1.96*se:.2f}, {mu+1.96*se:.2f}]"


METRIC_KEYS   = ["acc","bacc","f1","sens","spec","pre","npv","mcc","auc","ap"]
PERCLASS_KEYS = [f"{s}_c{i}" for i in range(3) for s in ("sens","spec","pre","npv")]


def new_store():
    d = {f"tr_{k}": [] for k in METRIC_KEYS}
    d.update({f"cv_{k}": [] for k in METRIC_KEYS + PERCLASS_KEYS})
    return d


def record(store, y_tr, y_ts, p_tr, p_ts, s_ts=None):
    a = get_metrics(y_tr, p_tr)
    b = get_metrics(y_ts, p_ts, s_ts)
    for k in METRIC_KEYS:
        store[f"tr_{k}"].append(a[k])
    for k in METRIC_KEYS + PERCLASS_KEYS:
        store[f"cv_{k}"].append(b[k])


def build_row(name, stage, store, best_solo=None, extra=None):
    tr  = np.mean(store["tr_acc"]) * 100
    cvv = np.mean(store["cv_acc"]) * 100
    row = {"Config": name, "Stage": stage}
    for k, lbl in [("acc","ACC"),("bacc","BACC"),("sens","SENS"),("spec","SPEC"),
                   ("pre","PRE"),("npv","NPV"),("f1","F1"),("mcc","MCC"),
                   ("auc","AUC"),("ap","AP")]:
        raw = k in ("mcc","auc","ap")
        row[f"Train_{lbl}"]   = fmt(store[f"tr_{k}"], raw)
        row[f"CV_{lbl}"]      = fmt(store[f"cv_{k}"], raw)
        row[f"CV_{lbl}_95CI"] = fmt_ci(store[f"cv_{k}"], raw)
    for i in range(3):
        cn = CLASS_NAMES[i]
        for s, lbl in [("sens","SENS"),("spec","SPEC"),("pre","PRE"),("npv","NPV")]:
            row[f"CV_{lbl}_{cn}"] = fmt(store[f"cv_{s}_c{i}"])
    row["CV_ACC_raw"]  = round(cvv, 4)
    row["CV_BACC_raw"] = round(np.mean(store["cv_bacc"]) * 100, 4)
    row["CV_F1_raw"]   = round(np.mean(store["cv_f1"]) * 100, 4)
    row["Overfit_Gap"] = round(tr - cvv, 2)
    if best_solo is not None:
        row["Best_Solo_ACC"] = round(best_solo, 2)
        row["Gain_vs_Best"]  = round(cvv - best_solo, 2)
    if extra:
        row.update(extra)
    return row

In [ ]:


def _objective(y_true, P, w, objective):
    pred = np.argmax(P * w[None, :], axis=1)
    if objective == "acc":
        return -accuracy_score(y_true, pred)
    if objective == "bacc":
        return -balanced_accuracy_score(y_true, pred)
    return -f1_score(y_true, pred, average="macro")


def fit_decision_weights(y_oof, P_oof, objective="acc", n_classes=3, seed=SEED):
    """Return w (n_classes,) maximising `objective` on out-of-fold train probs."""
    best_w, best_v = np.ones(n_classes), _objective(y_oof, P_oof, np.ones(n_classes), objective)
    rs = np.random.RandomState(seed)
    starts = [np.zeros(n_classes)] + [rs.normal(0, 0.35, n_classes) for _ in range(6)]
    for s in starts:
        r = minimize(lambda t: _objective(y_oof, P_oof, np.exp(t), objective),
                     s, method="Nelder-Mead",
                     options=dict(maxiter=600, xatol=1e-3, fatol=1e-5))
        if r.fun < best_v:
            best_v, best_w = r.fun, np.exp(r.x)
    return best_w / best_w.sum() * n_classes      # scale-free; printed in Methods


def apply_decision(P, w):
    return np.argmax(P * w[None, :], axis=1)

In [ ]:

def enrich(blocks):
    """Strict function of the base probabilities -- no new feature source."""
    P = np.hstack(blocks)
    if not ENRICH_META:
        return P
    extra = []
    for B in blocks:
        Bc = np.clip(B, 1e-9, 1.0)
        extra.append(Bc.max(axis=1, keepdims=True))                    # confidence
        extra.append(-(Bc * np.log(Bc)).sum(axis=1, keepdims=True))    # entropy
    votes = np.stack([np.argmax(B, axis=1) for B in blocks], axis=1)
    agree = np.stack([(votes == c).mean(axis=1) for c in range(3)], axis=1)
    return np.hstack([P] + extra + [agree])


def make_meta():
    return LogisticRegression(C=1.0, max_iter=3000, solver="lbfgs",
                              class_weight="balanced", random_state=SEED, n_jobs=-1)

In [ ]:

df = pd.read_csv(FEATURES_PATH)
missing = [c for c in META_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing metadata column(s): {missing}")

dim_cols  = [c for c in df.columns if c.startswith("dim_")]
X_raw     = df[dim_cols].to_numpy(np.float32)
y         = df[TARGET_COL].to_numpy()
y_strat   = df[STRAT_COL].to_numpy()
n_classes = int(len(np.unique(y)))

if X_raw.shape[1] < 500:
    raise ValueError("Use the FULL 1024-dim ProtT5 training file: the LGBM_gain "
                     "selection is refitted inside the folds here.")
for b in BASE_POOL:                        # fail fast, before hours of compute
    make_model(b)
print("All pool entries resolve to a factory entry.\n")

ALL_COMBOS = [c for s in COMBO_SIZES if s <= len(BASE_POOL)
              for c in combinations(BASE_POOL, s)]
maj = np.bincount(y).max() / len(y) * 100

print(f"X {X_raw.shape}   class counts {np.bincount(y)}")
print(f"Majority-class rate (the number any model must beat) : {maj:.2f}%")
print(f"Feature optimization : {SELECTION_METHOD}, k={SELECT_K} (refitted in-fold)")
print(f"Base pool            : {BASE_POOL}")
print(f"Combinations         : {len(ALL_COMBOS)}  (sizes {list(COMBO_SIZES)})")
print(f"Outer {N_OUTER}-fold stratified on {STRAT_COL} | inner {N_INNER}-fold")
print(f"Calibration={CALIBRATE}  enriched meta={ENRICH_META}  "
      f"decision objective={DECISION_OBJECTIVE}")
print("Locked test split    : NOT read by this notebook\n")

outer_cv = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)

In [ ]:


cache, t_all = [], time.time()

for fold, (tr_idx, ts_idx) in enumerate(outer_cv.split(X_raw, y_strat)):
    print(f"{'-'*70}\n  Outer fold {fold+1}/{N_OUTER}\n{'-'*70}")
    X_tr_o, X_ts_o = X_raw[tr_idx], X_raw[ts_idx]
    y_tr,   y_ts   = y[tr_idx],     y[ts_idx]
    s_tr           = y_strat[tr_idx]

    oof = {b: np.zeros((len(y_tr), n_classes)) for b in BASE_POOL}
    ins, tst = {}, {}

    inner_cv = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=SEED)
    for j, (i_tr, i_va) in enumerate(inner_cv.split(X_tr_o, s_tr)):
        t0 = time.time()
        A, B = select_lgbm_gain(X_tr_o[i_tr], X_tr_o[i_va], y_tr[i_tr])
        sc = StandardScaler().fit(A)
        A, B = sc.transform(A), sc.transform(B)
        for b in BASE_POOL:
            oof[b][i_va] = fit_predict_base(b, A, y_tr[i_tr], [B])[0]
        print(f"    inner {j+1}/{N_INNER} done  [{time.time()-t0:.0f}s]")

    t0 = time.time()
    A, B = select_lgbm_gain(X_tr_o, X_ts_o, y_tr)
    sc = StandardScaler().fit(A)
    A, B = sc.transform(A), sc.transform(B)
    for b in BASE_POOL:
        p_in, p_ts = fit_predict_base(b, A, y_tr, [A, B])
        ins[b], tst[b] = p_in, p_ts
    print(f"    outer fit done  [{time.time()-t0:.0f}s]")

    cache.append(dict(oof=oof, ins=ins, tst=tst, y_tr=y_tr, y_ts=y_ts))
    gc.collect()

print(f"\nPASS 1 complete in {(time.time()-t_all)/60:.1f} min")

In [ ]:

solo, solo_ll = {}, {}
for b in BASE_POOL:
    a = [accuracy_score(f["y_ts"], np.argmax(f["tst"][b], 1)) for f in cache]
    bc = [balanced_accuracy_score(f["y_ts"], np.argmax(f["tst"][b], 1)) for f in cache]
    ll = [log_loss(f["y_tr"], np.clip(f["oof"][b], 1e-9, 1),
                   labels=list(range(n_classes))) for f in cache]
    solo[b] = (np.mean(a) * 100, np.mean(bc) * 100)
    solo_ll[b] = np.mean(ll)

print(f"{'='*80}\n  SOLO BASE PERFORMANCE ({N_OUTER} outer folds)\n{'='*80}")
for b, (a, bc) in sorted(solo.items(), key=lambda kv: -kv[1][0]):
    print(f"  {b:<14} ACC {a:6.2f}   BACC {bc:6.2f}   OOF logloss {solo_ll[b]:.4f}")
BEST_SOLO_ACC = max(v[0] for v in solo.values())
print(f"\n  best single base : {BEST_SOLO_ACC:.2f}  "
      f"(every combination below must beat this)")


print(f"\n{'='*80}\n  PAIRWISE DISAGREEMENT BETWEEN BASES (held-out folds, %)\n{'='*80}")
dis_mat = pd.DataFrame(index=BASE_POOL, columns=BASE_POOL, dtype=float)
for bi in BASE_POOL:
    for bj in BASE_POOL:
        if bi == bj:
            dis_mat.loc[bi, bj] = 0.0
        else:
            d = [(np.argmax(f["tst"][bi], 1) != np.argmax(f["tst"][bj], 1)).mean()
                 for f in cache]
            dis_mat.loc[bi, bj] = np.mean(d) * 100
print(dis_mat.round(2).to_string())
dis_mat.to_csv(os.path.join(OUTPUT_DIR, "base_pairwise_disagreement.csv"))
print("\n  KNN_cosine's row is the one to read: if it disagrees with the trees")
print("  and the SVM substantially more than they disagree with each other, it")
print("  is contributing the diversity it was added for.")

In [ ]:

def eval_combo(combo):
    store, wlog = new_store(), []
    dis, ora = [], []
    for f in cache:
        y_tr, y_ts = f["y_tr"], f["y_ts"]
        M_tr = enrich([f["oof"][b] for b in combo])
        M_in = enrich([f["ins"][b] for b in combo])
        M_ts = enrich([f["tst"][b] for b in combo])

        meta = make_meta().fit(M_tr, y_tr)
        P_tr = np.zeros((len(y_tr), n_classes)); P_in = np.zeros_like(P_tr)
        P_ts = np.zeros((len(y_ts), n_classes))
        q_tr, q_in, q_ts = (meta.predict_proba(M_tr), meta.predict_proba(M_in),
                            meta.predict_proba(M_ts))
        for j, c in enumerate(meta.classes_):        # class-complete output
            P_tr[:, int(c)] = q_tr[:, j]
            P_in[:, int(c)] = q_in[:, j]
            P_ts[:, int(c)] = q_ts[:, j]

        w = fit_decision_weights(y_tr, P_tr, objective=DECISION_OBJECTIVE)
        wlog.append(w)
        record(store, y_tr, y_ts,
               apply_decision(P_in, w), apply_decision(P_ts, w), P_ts)

        votes = np.stack([np.argmax(f["tst"][b], 1) for b in combo], axis=1)
        prs = [(i, j) for i in range(len(combo)) for j in range(i+1, len(combo))]
        dis.append(np.mean([(votes[:, i] != votes[:, j]).mean() for i, j in prs]))
        ora.append(np.mean((votes == y_ts[:, None]).any(axis=1)))

    best_member = max(solo[b][0] for b in combo)
    return build_row("+".join(combo), "strategyA", store, best_solo=best_member,
                     extra={"N_Bases": len(combo),
                            "Has_KNN": "KNN_cosine" in combo,
                            "Mean_Disagreement": round(float(np.mean(dis))*100, 2),
                            "Oracle_ACC": round(float(np.mean(ora))*100, 2),
                            "Best_Member_ACC": round(best_member, 2),
                            "Decision_W": np.round(np.mean(wlog, 0), 3).tolist()})


print(f"\n{'='*80}\n  STRATEGY A : {len(ALL_COMBOS)} combinations\n{'='*80}")
rows = []
for c in ALL_COMBOS:
    r = eval_combo(list(c))
    rows.append(r)
    print(f"  n={r['N_Bases']}  ACC {r['CV_ACC_raw']:6.2f}  BACC {r['CV_BACC_raw']:6.2f}"
          f"  gain {r['Gain_vs_Best']:+5.2f}  dis {r['Mean_Disagreement']:5.1f}"
          f"  orc {r['Oracle_ACC']:5.1f}  | {'+'.join(c)}")

In [ ]:

res = pd.DataFrame(rows).sort_values("CV_ACC_raw", ascending=False).reset_index(drop=True)
res.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved : {OUTPUT_FILE}   shape {res.shape}")

cols = ["Config","N_Bases","Has_KNN","CV_ACC","CV_BACC","CV_F1","CV_MCC",
        "Gain_vs_Best","Mean_Disagreement","Oracle_ACC","Overfit_Gap"]
print(f"\n{'='*80}\n  ALL COMBINATIONS, RANKED\n{'='*80}")
print(res[cols].to_string(index=False))

print(f"\n{'='*80}\n  DID KNN_cosine EARN ITS PLACE?\n{'='*80}")
wk = res[res.Has_KNN]; nk = res[~res.Has_KNN]
print(f"  with KNN     n={len(wk):2d}  best ACC {wk.CV_ACC_raw.max():6.2f}  "
      f"mean {wk.CV_ACC_raw.mean():6.2f}  mean disagreement {wk.Mean_Disagreement.mean():5.2f}"
      f"  mean oracle {wk.Oracle_ACC.mean():5.2f}")
print(f"  without KNN  n={len(nk):2d}  best ACC {nk.CV_ACC_raw.max():6.2f}  "
      f"mean {nk.CV_ACC_raw.mean():6.2f}  mean disagreement {nk.Mean_Disagreement.mean():5.2f}"
      f"  mean oracle {nk.Oracle_ACC.mean():5.2f}")
print("""
  Read disagreement and oracle FIRST, accuracy second. KNN was added to raise
  diversity; if its combinations show higher disagreement and a higher oracle but
  the same accuracy, the diversity is real and the META-LEARNER is failing to
  convert it -- that is a notebook-3 problem (non-linear meta, restacking), not a
  reason to drop KNN. If disagreement does not move either, the representation is
  the ceiling and the member swap changed nothing.""")

print(f"\n{'='*80}\n  SPREAD CHECK -- is any difference here real?\n{'='*80}")
sd = res["CV_ACC_raw"].std()
print(f"  best {res.CV_ACC_raw.max():.2f}  worst {res.CV_ACC_raw.min():.2f}  "
      f"range {res.CV_ACC_raw.max()-res.CV_ACC_raw.min():.2f}  std {sd:.3f}")
print(f"  best combination 95% CI : {res.iloc[0]['CV_ACC_95CI']}")
if res.CV_ACC_raw.max() - res.CV_ACC_raw.min() < 0.5:
    print("  The spread is inside fold noise. Do not pick the argmax and call it")
    print("  the best model -- choose on diversity and parsimony, and say so in")
    print("  Methods. A 0.1-point 'win' is not a result.")

print(f"\n  accuracy / diversity by combination size:")
print(res.groupby("N_Bases").agg(best_ACC=("CV_ACC_raw","max"),
                                 mean_ACC=("CV_ACC_raw","mean"),
                                 mean_dis=("Mean_Disagreement","mean"),
                                 mean_oracle=("Oracle_ACC","mean")).round(2).to_string())

print(f"\n{'='*80}\n  DIAGNOSIS\n{'='*80}")
print(f"  majority-class rate      : {maj:.2f}")
print(f"  best single base         : {BEST_SOLO_ACC:.2f}")
print(f"  best stacked combination : {res.CV_ACC_raw.max():.2f}")
print(f"  best gain over member    : {res.Gain_vs_Best.max():+.2f}")
print(f"  highest oracle reached   : {res.Oracle_ACC.max():.2f}")
print(f"  stack-to-oracle gap      : {res.Oracle_ACC.max()-res.CV_ACC_raw.max():.2f}"
      "   <- the headroom notebook 3 targets")

BEST_COMBO = res.iloc[0]["Config"].split("+")
print(f"\n  -> carry into notebook 3 : {BEST_COMBO}")
json.dump({"best_combo": BEST_COMBO, "strategy": "A",
           "selection_method": SELECTION_METHOD, "select_k": SELECT_K,
           "cv_acc": float(res.iloc[0]["CV_ACC_raw"]),
           "cv_bacc": float(res.iloc[0]["CV_BACC_raw"]),
           "decision_objective": DECISION_OBJECTIVE,
           "calibrate": CALIBRATE, "enrich_meta": ENRICH_META,
           "n_outer": N_OUTER, "n_inner": N_INNER, "seed": SEED},
          open(os.path.join(OUTPUT_DIR, "best_base_combo.json"), "w"), indent=2)
print("  Written : best_base_combo.json")